# Claude Code가 User-Agent 헤더에 계정 이메일을 넣어 보낸다는 이슈 정리: 어떤 상황에서 생기는지, 2.1.317 보호 기능과 제보된 우회 사례, 내 세션 기록 점검법

위에서부터 차례로 실행해 보세요.

### 버전 확인

In [ ]:
%%bash
mkdir -p cc && cd cc && npm i @anthropic-ai/claude-code@2.1.319 2>&1 | tail -3
./node_modules/.bin/claude --version
npm view @anthropic-ai/claude-code dist-tags --json

### 합성 JSONL 만들기

In [ ]:
%%bash
mkdir -p syn/proj/-work-demo/s1/subagents && cd syn
cat > proj/-work-demo/s1.jsonl <<'E'
{"type":"user","uuid":"u1","timestamp":"2026-11-01T09:00:00Z","message":{"role":"user","content":"# userEmail\nThe user's email address is test.user@example.com.\nlook up Liquipedia"}}
{"type":"assistant","uuid":"a1","timestamp":"2026-11-01T09:00:05Z","message":{"role":"assistant","content":[{"type":"tool_use","id":"toolu_01","name":"Bash","input":{"command":"curl -s -A \"research-bot (test.user@example.com)\" https://liquipedia.net/api.php?action=query","description":"fetch"}}]}}
{"type":"user","uuid":"u2","timestamp":"2026-11-01T09:00:07Z","message":{"role":"user","content":[{"type":"tool_result","tool_use_id":"toolu_01","content":"HTTP 403 Forbidden","is_error":true}]}}
{"type":"assistant","uuid":"a2","timestamp":"2026-11-01T09:01:00Z","message":{"role":"assistant","content":[{"type":"tool_use","id":"toolu_02","name":"Bash","input":{"command":"curl -s https://en.wikipedia.org/w/api.php -H 'User-Agent: kb-tool/1.0'"}},{"type":"tool_use","id":"toolu_03","name":"Write","input":{"file_path":"/tmp/n.txt","content":"owner: test.user@example.com"}}]}}
{"type":"user","uuid":"u3","timestamp":"2026-11-01T09:01:02Z","message":{"role":"user","content":[{"type":"tool_result","tool_use_id":"toolu_02","content":"ok"},{"type":"tool_result","tool_use_id":"toolu_03","content":"written"}]}}
E
cat > proj/-work-demo/s1/subagents/agent-x.jsonl <<'E'
{"type":"assistant","uuid":"s1","timestamp":"2026-11-01T10:00:00Z","message":{"role":"assistant","content":[{"type":"tool_use","id":"toolu_09","name":"Bash","input":{"command":"curl -A \"Mozilla (TEST.USER@EXAMPLE.COM)\" https://en.wikipedia.org/w/index.php?search=x"}}]}}
{"type":"user","uuid":"s2","timestamp":"2026-11-01T10:00:03Z","message":{"role":"user","content":[{"type":"tool_result","tool_use_id":"toolu_09","content":"The tool call was denied: account email in User-Agent","is_error":true}]}}
E

### 점검 스크립트 만들기

In [ ]:
%%bash
cd syn
cat > audit.py <<'E'
#!/usr/bin/env python3
import sys, json, re, pathlib, urllib.parse
email, root = sys.argv[1], pathlib.Path(sys.argv[2])
needles = {email.lower(), urllib.parse.quote(email, safe='').lower()}
UA = re.compile(r"user[-_]agent|\s-A\s|\s-U\s|-UserAgent|\bUA=", re.I)
URL = re.compile(r"https?://[^\s'\"\\)]+")
def walk(f):
    for line in f.read_text(errors='replace').splitlines():
        try: yield json.loads(line)
        except json.JSONDecodeError: pass
def has(s): s=s.lower(); return any(n in s for n in needles)
hits=[]; others=0
for f in sorted(root.rglob('*.jsonl')):
    recs=list(walk(f)); results={}
    for r in recs:
        c=(r.get('message') or {}).get('content')
        if isinstance(c,list):
            for b in c:
                if b.get('type')=='tool_result': results[b.get('tool_use_id')]=b
    for r in recs:
        c=(r.get('message') or {}).get('content')
        if isinstance(c,str):
            if has(c): others+=1
            continue
        for b in c or []:
            if b.get('type')=='tool_use':
                inp=json.dumps(b.get('input'),ensure_ascii=False)
                if has(inp):
                    res=results.get(b['id'],{})
                    hits.append(dict(file=str(f.relative_to(root)),time=r.get('timestamp'),tool=b['name'],ua=bool(UA.search(inp)),urls=URL.findall(inp),err=res.get('is_error'),result=str(res.get('content','(no result)'))[:60]))
            elif b.get('type')=='text' and has(b.get('text','')): others+=1
print(f"{'time':20} {'tool':6} UA  {'result':8} file / urls")
for h in hits:
    print(f"{h['time']:20} {h['tool']:6} {'Y' if h['ua'] else '-'}   {'error' if h['err'] else 'ok':8} {h['file']} {h['urls']} | {h['result']}")
print(f"\ntool_use hits: {len(hits)} (with UA: {sum(h['ua'] for h in hits)}); mentions elsewhere (context/text): {others}")
E

### 합성 기록에 점검 실행

In [ ]:
%%bash
cd syn && python3 -I audit.py test.user@example.com proj